# StreamFlix Content Analytics - Phase 1: Data Loading, Cleaning & Quality Report

### **Analyst**: Eram Aiysha Shaikh
### **Date**: 05/07/2026

**Objective**: Load all 6 CSV files, profile each, and run data quality and data integrity checks required by the project before any analysis begins.
All the findings are documented with explanation of what it means and why it matters.

### Step 1 - Load all 6 CSV files:
Before doing anything else, we load each of the 6 CSV files into its own DataFrame
and check three things for each: **row count, column count, and data types**.

This matters because:
- A wrong row count (e.g. `watch_history` loading with far fewer than 650,000 rows)
  signals a load problem before we waste time analyzing bad data.
- pandas reads dates as plain text (`object` dtype) unless told otherwise — we need to
  know this now so we can fix it in a later step, before it breaks any trend chart.This matters because:
- A wrong row count (e.g. `watch_history` loading with far fewer than 650,000 rows)
  signals a load problem before we waste time analyzing bad data.
- pandas reads dates as plain text (`object` dtype) unless told otherwise — we need to
  know this now so we can fix it in a later step, before it breaks any trend chart.

In [2]:
import pandas as pd
import numpy as np

#load all 6 csv
DATA_DIR = "../data/"
#this will read all the tables
ratings = pd.read_csv(DATA_DIR + 'ratings.csv')
reviews = pd.read_csv(DATA_DIR + 'reviews.csv')
subscribers = pd.read_csv(DATA_DIR + 'subscribers.csv')
titles = pd.read_csv(DATA_DIR + 'titles.csv')
watch_history = pd.read_csv(DATA_DIR + 'watch_history.csv')
watchlist = pd.read_csv(DATA_DIR + 'watchlist.csv')



In [3]:
#putting all the tables in a dictionary
tables = {
    'ratings':ratings,
    'reviews':reviews,
    'subscribers':subscribers,
    'titles':titles,
    'watch_history':watch_history,
    'watchlist':watchlist
}

for name,df in tables.items():  #this will go through each name of the table and col of those table in the tables dict
    print(f'{name}: {df.shape[0]:,} rows x {df.shape[1]} columns')#this will print the table name and its rows and cols count

ratings: 130,000 rows x 5 columns
reviews: 110,000 rows x 7 columns
subscribers: 15,000 rows x 14 columns
titles: 9,000 rows x 21 columns
watch_history: 650,000 rows x 10 columns
watchlist: 65,000 rows x 5 columns


In [4]:
# Look at data types for each table — this tells us which columns need converting
for name,df in tables.items():  #this will check the name of the table and each col of that table present in the tables dict
    print(f'\n-----{name} dtypes-----') #this will print the name of the table
    print(df.dtypes)            #this will print the datatypes of each col of the table


-----ratings dtypes-----
rating_id        int64
subscriber_id      str
title_id           str
rating           int64
rating_date        str
dtype: object

-----reviews dtypes-----
review_id        int64
subscriber_id      str
title_id           str
review_text        str
sentiment          str
helpful_votes    int64
review_date        str
dtype: object

-----subscribers dtypes-----
subscriber_id            str
signup_date              str
country                  str
region                   str
age                    int64
gender                   str
plan_type                str
monthly_price_usd    float64
household_size         int64
primary_device           str
payment_method           str
tenure_months          int64
is_active               bool
churn_date               str
dtype: object

-----titles dtypes-----
title_id                    str
title_name                  str
type                        str
primary_genre               str
country                     str
language 

### Finding 1 — Row, column counts and data types

- All 6 tables loaded with the expected row counts described in the project brief
  (subscribers: 15,000 · titles: 9,000 · watch_history: 650,000 · ratings: 130,000 ·
  reviews: 110,000 · watchlist: 65,000).
- Date columns (`signup_date`, `churn_date`, `watch_date`, `rating_date`, `review_date`,
  `added_date`, `date_added`, `license_expiry`) are currently loaded as `object`
  (text), not proper dates. These will be converted in the next step.  

### Step 2 - Check for missing values and duplicate records

**This matters because**:

Missing values can mean two different things: a genuine data problem, or an expected 
gap (e.g. a subscriber who never churned has no churn date - that's not an error). 
Duplicate primary keys (`subscriber_id`, `title_id`, `watch_id`) would badly distort 
every KPI calculated later, so these are checked before anything else.

In [6]:
for name,df in tables.items():  #the name of each col & the col from the tables dict above
    null = df.isnull().sum()    #null contains all the col's missing value count eg: ratings table conatins rating_id,subscriber_id col etc. it calcutes and store it in null
    print(f'\n{null}')          #this part is printing all the cols with missing value couunt be it 0 or not
    null=null[null>0]           #this line stores only those value count which is > then 0 eg: if ratings tables cols have value > 0 then it is stored in null if not then the null will be empty
    if len(null)>0:             #this checks if length of null > 0 or not eg: for ratings table all the col have 0 missing value count, so null be empty, so it will go to the else part directly
        print(f'\n{name} - missing values')
        
        for col,count in null.items():  #this will check the cols and count of the table present in null
            print(f'\n{col} - {count:,} missing values ({count/len(df)*100:.1f}%)') #print missing values and percent
            print('--------------------------')
    else:
         print(f'\n{name} - no missing values') 
         print('--------------------------')      


rating_id        0
subscriber_id    0
title_id         0
rating           0
rating_date      0
dtype: int64

ratings - no missing values
--------------------------

review_id        0
subscriber_id    0
title_id         0
review_text      0
sentiment        0
helpful_votes    0
review_date      0
dtype: int64

reviews - no missing values
--------------------------

subscriber_id            0
signup_date              0
country                  0
region                   0
age                      0
gender                   0
plan_type                0
monthly_price_usd        0
household_size           0
primary_device           0
payment_method           0
tenure_months            0
is_active                0
churn_date           11199
dtype: int64

subscribers - missing values

churn_date - 11,199 missing values (74.7%)
--------------------------

title_id                   0
title_name                 0
type                       0
primary_genre              0
country               

Checking duplicate row:


In [9]:
#We can check if there is any duplicate primary key in all the tables
print('Duplicate subscriber_id:',subscribers['subscriber_id'].duplicated().sum())#this line will check for duplicate record
print('Duplicate watch_id:',watch_history['watch_id'].duplicated().sum())
print('Duplicate title_id:',titles['title_id'].duplicated().sum())
print('Duplicate watchlist_id:',watchlist['watchlist_id'].duplicated().sum())
print('Duplicate review_id:',reviews['review_id'].duplicated().sum())
print('Duplicate rating_id:',ratings['rating_id'].duplicated().sum())

   


Duplicate subscriber_id: 0
Duplicate watch_id: 0
Duplicate title_id: 0
Duplicate watchlist_id: 0
Duplicate review_id: 0
Duplicate rating_id: 0


### Finding 2 - Missing values and duplicates

- `subscribers.churn_date` is missing in 74.7% of rows - this is expected, not an 
  error, since `churn_date` is only populated for subscribers who have actually 
  churned. Confirmed in Step 4 that every missing `churn_date` lines up exactly with 
  `is_active = True`.
- `titles.license_expiry` is missing in 21.8% of rows - also expected, since 
  StreamFlix Originals don't have a license expiry (only licensed third-party content 
  does).
- No other columns in any of the 6 tables have missing values.
- Zero duplicate `subscriber_id`, `title_id`,  `watch_id` , `watchlist` , `review_id` , `rating_id` - each primary key is 
  unique, so no records will be double-counted in later KPI calculations.

### Step 3 - Convert date columns to proper datetime type

Every date column identified in Step 1 is converted from text to a real `datetime` 
type. Without this, sorting by date, calculating tenure, or building a monthly trend 
chart in Phase 2 would either fail or silently give wrong results.

In [12]:
#Converting all the date columns to proper datetime datatype
subscribers['signup_date']=pd.to_datetime(subscribers['signup_date'])#this line convert the date object to datatime datatype
subscribers['churn_date']=pd.to_datetime(subscribers['churn_date'])

ratings['rating_date']=pd.to_datetime(ratings['rating_date'])

reviews['review_date']=pd.to_datetime(reviews['review_date'])

watch_history['watch_date']=pd.to_datetime(watch_history['watch_date'])

watchlist['added_date']=pd.to_datetime(watchlist['added_date'])

titles['license_expiry'] = pd.to_datetime(titles['license_expiry'])

print("added_date:", watchlist['added_date'].dtype)
print("churn_date:", subscribers['churn_date'].dtype)
print("rating_date:", ratings['rating_date'].dtype)
print("review_date:", reviews['review_date'].dtype)
print("license_expiry:", titles['license_expiry'].dtype)
print("signup_date:", subscribers['signup_date'].dtype)
print("watch_date:", watch_history['watch_date'].dtype)
print("\nsignup_date range:", subscribers['signup_date'].min(), "to", subscribers['signup_date'].max())
print("watch_date range:", watch_history['watch_date'].min(), "to", watch_history['watch_date'].max())


added_date: datetime64[us]
churn_date: datetime64[us]
rating_date: datetime64[us]
review_date: datetime64[us]
license_expiry: datetime64[us]
signup_date: datetime64[us]
watch_date: datetime64[us]

signup_date range: 2016-01-13 00:00:00 to 2026-05-31 00:00:00
watch_date range: 2017-03-10 00:00:00 to 2026-05-31 00:00:00


### Finding 3 - Date conversion

- All date columns successfully converted from text to `datetime64`.
- `signup_date` ranges from 2016-01-13 to 2026-05-31 - just over 10 years of 
  subscriber history.
- `watch_date` ranges from 2017-03-10 to 2026-05-31, consistent with viewing activity 
  starting after the platform's earliest signups (as expected - customers sign up 
  before they start watching).

### Step 4 - Business logic and referential integrity checks

Beyond generic null/duplicate checks, the brief requires validating that the data 
makes business sense: churn dates should never precede signup dates, no one should 
watch more minutes than a title's actual runtime, `completion_pct` should match the 
watch time it's calculated from, and every `subscriber_id`/`title_id` referenced in 
`watch_history` should actually exist in the parent tables.

In [27]:
#Check 1: churn_date should never be before signup_date
churn_date_check = subscribers[(subscribers['churn_date'].notna()) & (subscribers['churn_date'] < subscribers['signup_date'])]#this line will check whether the churn_date not null and churn date is newer then the sign up date
print(f'Churn date should never precede signup dates (should be 0):',len(churn_date_check))

#Check 2:subscriber is active or not and if it is active then whether the churn date is set(churn date should be null because the subscriber is active)
active_subscriber = subscribers[(subscribers['is_active']==True) & (subscribers['churn_date'].notna())]#this line will check whether the subscribers is active col is st to true and has the subscriber set the churn date which is not null
print("Active subscribers with a churn_date set (should be 0):", len(active_subscriber))

#Check 3:subscriber is active or not and if it is inactive then whether the churn date is set(churn date should not be null because the subscriber is inactive)
inactive_subscriber = subscribers[(subscribers['is_active']==False) & (subscribers['churn_date'].isna())]#this will check whether the is active is set to false and check whether the churn date is missed or not
print("Inactive subscribers missing a churn_date (should be 0):", len(inactive_subscriber))

#Check 4:invalid watch hours:watch_duration_min should never exceed content_duration_min
invalid_watch_time = watch_history[watch_history['watch_duration_min'] > watch_history['content_duration_min']]#this checks whether the watch duration > then the actual series/movies duration
print(f'Watch time longer then the actual content duration (should be 0):',len(invalid_watch_time))

#Check 5:Calculate the completion time and check if the completion pct matches the calculated time
watch_history['calc_completion'] = (watch_history['watch_duration_min']/watch_history['content_duration_min'])*100#this will calculate the percent of watch time according to watch_duration and content_duration
watch_history['completion_diff'] = watch_history['completion_pct'] - watch_history['calc_completion']#this will calculate the difference between given completion percent and calculated percent
mismatched_completion = watch_history[watch_history['completion_diff'].abs() > 5]#this will check the mismatch values if > 5
print("Rows where completion_pct differs from the calculated value by >5 points:", len(mismatched_completion))

#Check 6:every subscriber_id/title_id reference in watch_history should actually exist in parent tables
#here we will check for orphaned ids: an orphaned records are those whole refernec is not present in the connected table, like they are lost or left with connection
orphaned_subscribers_watch_history = (~watch_history['subscriber_id'].isin(subscribers['subscriber_id'])).sum()#this line will check the total of all the watch_history ids which is not present in subscribers ids
orphaned_titles_watch_history = (~watch_history['title_id'].isin(titles['title_id'])).sum()
print("watch_history rows referencing an unknown subscriber_id (should be 0):", orphaned_subscribers_watch_history)
print("watch_history rows referencing an unknown title_id (should be 0):", orphaned_titles_watch_history)

orphaned_subscribers_watchlist = (~watchlist['subscriber_id'].isin(subscribers['subscriber_id'])).sum()
orphaned_titles_watchlist = (~watchlist['title_id'].isin(titles['title_id'])).sum()
print("watchlist rows referencing an unknown subscriber_id (should be 0):", orphaned_subscribers_watchlist)
print("watchlist rows referencing an unknown title_id (should be 0):", orphaned_titles_watchlist)

orphaned_subscribers_ratings = (~ratings['subscriber_id'].isin(subscribers['subscriber_id'])).sum()
orphaned_titles_ratings = (~ratings['title_id'].isin(titles['title_id'])).sum()
print("ratings rows referencing an unknown subscriber_id (should be 0):", orphaned_subscribers_ratings)
print("ratings rows referencing an unknown title_id (should be 0):", orphaned_titles_ratings)

orphaned_subscribers_reviews = (~reviews['subscriber_id'].isin(subscribers['subscriber_id'])).sum()
orphaned_titles_reviews = (~reviews['title_id'].isin(titles['title_id'])).sum()
print("reviews rows referencing an unknown subscriber_id (should be 0):", orphaned_subscribers_reviews)
print("reviews rows referencing an unknown title_id (should be 0):", orphaned_titles_reviews)

#Check 7: check for all the categorical value counts
print("\nreviews.sentiment unique values:")
print(reviews['sentiment'].value_counts())

print("\nsubscribers.gender unique values:")
print(subscribers['gender'].value_counts())

print("\nsubscribers.plan_tyoe unique values:")
print(subscribers['plan_type'].value_counts())

print("\ntitles.type unique values:")
print(titles['type'].value_counts())

print("\ntitles.primary_genre unique values:")
print(titles['primary_genre'].value_counts())

print("\nwatch_history.device unique values:")
print(watch_history['device'].value_counts())




Churn date should never precede signup dates (should be 0): 0
Active subscribers with a churn_date set (should be 0): 0
Inactive subscribers missing a churn_date (should be 0): 0
Watch time longer then the actual content duration (should be 0): 0
Rows where completion_pct differs from the calculated value by >5 points: 0
watch_history rows referencing an unknown subscriber_id (should be 0): 0
watch_history rows referencing an unknown title_id (should be 0): 0
watchlist rows referencing an unknown subscriber_id (should be 0): 0
watchlist rows referencing an unknown title_id (should be 0): 0
ratings rows referencing an unknown subscriber_id (should be 0): 0
ratings rows referencing an unknown title_id (should be 0): 0
reviews rows referencing an unknown subscriber_id (should be 0): 0
reviews rows referencing an unknown title_id (should be 0): 0

reviews.sentiment unique values:
sentiment
Positive    75007
Neutral     22759
Negative    12234
Name: count, dtype: int64

subscribers.gender u

### Finding 4 - Business logic and referential integrity

All checks passed with zero violations:

- `is_active` and `churn_date` are perfectly consistent - every active subscriber has 
  no churn date, and every inactive subscriber has one. No mismatches.
- No churn date occurs before its subscriber's signup date.
- No watch session exceeds the actual runtime of the title being watched - all 
  `watch_duration_min` values are logically valid.
- `completion_pct` matches the value calculated from `watch_duration_min / 
  content_duration_min` in every single row - the field can be trusted as-is for 
  KPI calculations in Phase 3.
- Zero orphaned `subscriber_id` or `title_id` values in `watch_history` - referential 
  integrity across the star schema is fully intact, so joins in Phase 2/3 won't 
  silently drop or duplicate rows.
- `reviews.sentiment` only contains the three expected categories: Positive (75,007), 
  Neutral (22,759), Negative (12,234) - no stray or misspelled category values.
- All the other categories are also calculated like device,plan_type,primary_genre etc and no stray or misspelled category values.


**Overall conclusion:** this dataset is unusually clean. The two "missing value" 
columns (`churn_date`, `license_expiry`) are missing by design, not by error, and 
every business-logic and referential-integrity check passed with no violations. No 
row deletion or correction was required in this dataset - the main data-cleaning work 
here was type conversion (Step 3) and verification (Steps 2 and 4), and the findings 
below form the Data Quality Report itself.

### Step 5 - Data Quality Report (Summary)

| Check | Result |
|---|---|
| Row/column counts match brief | Pass - all 6 tables |
| Missing values | 2 expected nulls only (`churn_date`, `license_expiry`) |
| Duplicate primary keys | 0 across `subscriber_id`, `title_id`, `watch_id` |
| Date columns converted | Pass - all 7 date columns now `datetime64` |
| `is_active` vs `churn_date` consistency | Pass - 0 mismatches |
| `churn_date` after `signup_date` | Pass - 0 violations |
| `watch_duration_min` <= `content_duration_min` | Pass - 0 violations |
| `completion_pct` matches calculated value | Pass - 0 mismatches beyond 5pt tolerance |
| Referential integrity (`watch_history` to `subscribers`/`titles`) | Pass - 0 orphaned rows |
| `sentiment` category values | Pass - only Positive/Neutral/Negative present |

**Conclusion:** The StreamFlix dataset passed every data quality and integrity check 
required for Phase 1. No records required deletion or correction. The dataset is 
ready to move into Phase 2 (Exploratory Data Analysis).

# -----------------END OF PHASE 1-------------------